# GDG UAM ML Contest 2026 — Comprobación desde un clon limpio

Este cuaderno comprueba un clon nuevo de la entrega pública con el harness oficial y las 400 muestras del demo público. No entrena el modelo.

In [1]:
from pathlib import Path
import subprocess
import sys

SUBMISSION = Path("/content/gdg-uam-submission-check")
MATERIALS = Path("/content/ml-contest-materials")

if SUBMISSION.exists() or MATERIALS.exists():
    raise RuntimeError("Use a fresh Colab runtime for this clean-clone check.")

subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/DavidTorro/gdg-uam-ml-contest-2026.git",
     str(SUBMISSION)],
    check=True,
)
subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/GDG-UAM/ml-contest-materials.git",
     str(MATERIALS)],
    check=True,
)

revision = subprocess.check_output(
    ["git", "-C", str(SUBMISSION), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
weight = SUBMISSION / "models" / "exp033.sesgo.onnx"
assert weight.is_file() and weight.stat().st_size > 100_000

print(f"Submission revision: {revision}")
print(f"ONNX weight: {weight.stat().st_size:,} bytes")

Submission revision: 3bbc56d
ONNX weight: 225,795 bytes


## 2. Instalar dependencias de inferencia

Instala las versiones fijadas en la entrega y comprueba que las librerías de inferencia se importan correctamente.

In [2]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(SUBMISSION / "requirements.txt")],
    check=True,
)

subprocess.run(
    [sys.executable, "-c",
     'import numpy, numba, onnxruntime; '
     'print("NumPy", numpy.__version__, '
     '"| Numba", numba.__version__, '
     '"| ONNX Runtime", onnxruntime.__version__)'],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-c', 'import numpy, numba, onnxruntime; print("NumPy", numpy.__version__, "| Numba", numba.__version__, "| ONNX Runtime", onnxruntime.__version__)'], returncode=0)

## 3. Descargar y verificar el demo público

Descarga las 400 muestras del demo oficial y comprueba su SHA-256.

In [3]:
import hashlib

versions = subprocess.check_output(
    [sys.executable, "-c",
     'import numpy, numba, onnxruntime; '
     'print("NumPy", numpy.__version__, '
     '"| Numba", numba.__version__, '
     '"| ONNX Runtime", onnxruntime.__version__)'],
    text=True,
)
print(versions.strip())

DEMO = MATERIALS / "data" / "test_demo.npz"
DEMO.parent.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["curl", "-fL", "--retry", "3", "-o", str(DEMO),
     "https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/test_demo.npz"],
    check=True,
)

digest = hashlib.sha256(DEMO.read_bytes()).hexdigest()
expected = "03fce51b3dacfaaa04418c4907def59874c4b6963c0561c9ccac83bbca90aac0"
assert digest == expected, f"Unexpected demo checksum: {digest}"

print(f"Demo verified: {DEMO.stat().st_size:,} bytes")
print(f"SHA-256: {digest}")

NumPy 2.5.4 | Numba 0.68.1 | ONNX Runtime 1.21.1
Demo verified: 1,654,340 bytes
SHA-256: 03fce51b3dacfaaa04418c4907def59874c4b6963c0561c9ccac83bbca90aac0


## 4. Ejecutar el harness oficial

Evalúa el modelo clonado sobre las 400 muestras distintas del demo público. El harness se encarga de la inicialización y el calentamiento.

In [4]:
from pathlib import Path
import subprocess
import sys

SUBMISSION = Path("/content/gdg-uam-submission-check")
MATERIALS = Path("/content/ml-contest-materials")

if not (SUBMISSION / "model.py").is_file():
    raise RuntimeError("Missing submission clone: run the clone cell first.")
if not (MATERIALS / "harness.py").is_file():
    raise RuntimeError("Missing official harness: run the clone cell first.")
if not (MATERIALS / "data" / "test_demo.npz").is_file():
    raise RuntimeError("Missing public demo: run the download cell first.")

subprocess.run(
    [sys.executable, "harness.py", "evaluate",
     "--model", str(SUBMISSION / "model.py"),
     "--source", "file:data/test_demo.npz",
     "--runs", "1"],
    cwd=MATERIALS,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', 'harness.py', 'evaluate', '--model', '/content/gdg-uam-submission-check/model.py', '--source', 'file:data/test_demo.npz', '--runs', '1'], returncode=0)

## 5. Verificar el resultado

Lee el informe JSON del harness oficial y comprueba el resultado esperado en el demo público.

In [5]:
from pathlib import Path
import json
import subprocess
from IPython.display import Markdown, display

materials = Path("/content/ml-contest-materials")
submission = Path("/content/gdg-uam-submission-check")
report_path = materials / "results" / "model" / "run_0.json"
report = json.loads(report_path.read_text())

revision = subprocess.check_output(
    ["git", "-C", str(submission), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

scored = report["n_scored"]
correct = report["n_correct"]
accuracy = 100 * report["accuracy_all"]
score = report["score"]
latency_ms = 1000 * report["latency_s"]["mean"]

passed = (
    report["status"] == "ok"
    and scored == 400
    and correct == 338
    and abs(score - 3.7789064859078927) < 0.001
)

accuracy_es = f"{accuracy:.1f}".replace(".", ",")
score_es = f"{score:.6f}".replace(".", ",")
latency_es = f"{latency_ms:.4f}".replace(".", ",")

display(Markdown(f"""
### {'✅ CORRECTO' if passed else '❌ REVISAR'} · demo público

| Comprobación | Resultado |
| --- | ---: |
| Commit de entrega | `{revision}` |
| Muestras evaluadas | {scored} |
| Aciertos | {correct} |
| Precisión | {accuracy_es} % |
| Puntuación del demo | {score_es} |
| Latencia media de `predict` | {latency_es} ms |

Informe detallado: `{report_path}`
"""))

if not passed:
    raise AssertionError("El resultado del demo difiere de los 338/400 esperados.")


### ✅ CORRECTO · demo público

| Comprobación | Resultado |
| --- | ---: |
| Commit de entrega | `3bbc56d` |
| Muestras evaluadas | 400 |
| Aciertos | 338 |
| Precisión | 84,5 % |
| Puntuación del demo | 3,778906 |
| Latencia media de `predict` | 0,2284 ms |

Informe detallado: `/content/ml-contest-materials/results/model/run_0.json`
